In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report
from model import GraphSAGE

c:\Users\tusha\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint = torch.load('models/task_a_model.pt', weights_only=False)
data = checkpoint['data']
concept_id_map = checkpoint['concept_id_map']

In [3]:
concept_prereqs = pd.read_csv('datasets/concept_prerequisites.csv')

real_edges = set()
for _, row in concept_prereqs.iterrows():
    src = concept_id_map[row['concept_id']]
    dst = concept_id_map[row['prereq_concept_id']]
    real_edges.add((src, dst))

pos_src = []
pos_dst = []
for _, row in concept_prereqs.iterrows():
    src = concept_id_map[row['concept_id']]
    dst = concept_id_map[row['prereq_concept_id']]
    pos_src.append(src)
    pos_dst.append(dst)

neg_src = []
neg_dst = []
for _ in range(len(pos_src) * 10):
    while True:
        i = np.random.randint(0, 80)
        j = np.random.randint(0, 80)
        if i != j and (i, j) not in real_edges:
            neg_src.append(i)
            neg_dst.append(j)
            break

print(f"Positive pairs: {len(pos_src)}")
print(f"Negative pairs: {len(neg_src)}")

Positive pairs: 89
Negative pairs: 890


In [4]:
all_src = pos_src + neg_src
all_dst = pos_dst + neg_dst
all_labels = [1] * len(pos_src) + [0]*len(neg_src)

src_tensor = torch.tensor(all_src, dtype = torch.long)
dst_tensor = torch.tensor(all_dst, dtype=torch.long)
labels = torch.tensor(all_labels, dtype = torch.float)

print(f"Total pairs: {len(all_labels)}")
print(f"Positives: {sum(all_labels)}, Negatives: {len(all_labels) - sum(all_labels)}")

np.random.seed(42)
indices = np.arange(len(all_labels))
np.random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

Total pairs: 979
Positives: 89, Negatives: 890
Train: 783, Test: 196


In [5]:
model = GraphSAGE(256)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
with torch.no_grad():
    out = model(data)
    concept_embs = out[2]

print("Concept embeddings shape:", concept_embs.shape)

Concept embeddings shape: torch.Size([80, 256])


In [6]:
class LinkPredictor(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, src_emb, dst_emb):
        pair = torch.concat([src_emb, dst_emb, src_emb - dst_emb, src_emb * dst_emb], dim = 1)
        return self.net(pair).squeeze()
    
link_model = LinkPredictor(emb_dim=256)
print(link_model)

LinkPredictor(
  (net): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [7]:
optimizer = torch.optim.Adam(link_model.parameters(),lr = 0.001)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(1000):
    link_model.train()
    optimizer.zero_grad()

    train_src_emb = concept_embs[src_tensor[train_idx]]
    train_dst_emb = concept_embs[dst_tensor[train_idx]]
    train_labels = labels[train_idx]

    preds = link_model(train_src_emb, train_dst_emb)
    loss = loss_fn(preds, train_labels)

    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.7155
Epoch 200 | Loss: 0.0548
Epoch 400 | Loss: 0.0039
Epoch 600 | Loss: 0.0014
Epoch 800 | Loss: 0.0004


In [8]:
link_model.eval()
with torch.no_grad():
    test_src_emb = concept_embs[src_tensor[test_idx]]
    test_dst_emb = concept_embs[dst_tensor[test_idx]]
    test_labels = labels[test_idx]
    
    # Basic accuracy first
    preds = link_model(test_src_emb, test_dst_emb)
    pred_labels = (torch.sigmoid(preds) > 0.5).float()
    accuracy = (pred_labels == test_labels).float().mean()
    print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9133


In [9]:
link_model.eval()
with torch.no_grad():
    hits_at_3 = 0
    mrr_sum = 0
    total = 0

    for _, row in concept_prereqs.iterrows():
        src = concept_id_map[row['concept_id']]
        true_dst = concept_id_map[row['prereq_concept_id']]

        src_emb = concept_embs[src].unsqueeze(0).repeat(80, 1)
        all_dst_emb = concept_embs

        scores = link_model(src_emb, all_dst_emb)
        scores = torch.sigmoid(scores)

        ranked = scores.argsort(descending=True)

        rank = (ranked == true_dst).nonzero(as_tuple = True)[0].item() + 1

        if rank <=3:
            hits_at_3 +=1
        mrr_sum += 1.0 /rank
        total += 1

    print(f"Hits@3: {hits_at_3/total:.4f} ({hits_at_3}/{total})")
    print(f"MRR: {mrr_sum/total:.4f}")

Hits@3: 0.7753 (69/89)
MRR: 0.6135
